GEETA 4B — exact maths tutoring for typed / photographed questions (2026-10-13).

Found by a hallucination eval (40 scripted kid conversations against the real model):
when the language model wrote the hints and judged answers it said things like
"14 and 9 make more than 14 but less than 20", locked "$1 + 50c" in as 60 cents,
praised wrong answers and suggested "jump by 4 from 1" for sharing 20 between 4.

So for every common Year 1-3 question shape, Python now reads the question,
computes the answer, plans small steps, judges every number the kid says and
writes every reply (in the kid's world-buddy voice). The model is only used for
reading photos and for chat it can't get wrong - and that is checked too
(tools/tutor_engine `_gate_reply`). Unknown question shapes return None and
fall back to the old verified-plan path.


## Bootstrap

This notebook is a GEETA runtime module. Executed standalone (e.g. `jupyter execute tools/verifier.ipynb`),
the next cell registers the notebook-as-module loader so `geeta.*` imports resolve to `tools/*.ipynb`.
When imported as `geeta.<name>` via `tools/_nbmodule.ipynb`, the same cell runs (idempotent).


In [ ]:
# --- GEETA notebook-module bootstrap (idempotent; makes tools/*.ipynb importable as geeta.*) ---
import json as _json
import pathlib as _pathlib

_ROOT = _pathlib.Path.cwd()
if not (_ROOT / "tools" / "_nbmodule.ipynb").exists():      # run from project root or tools/
    _ROOT = _ROOT.parent
_nb = _json.loads((_ROOT / "tools" / "_nbmodule.ipynb").read_text())
_src = "".join("".join(c["source"]) for c in _nb["cells"] if c["cell_type"] == "code")
exec(compile(_src, str(_ROOT / "tools" / "_nbmodule.ipynb"), "exec"), globals())
bootstrap(_ROOT)  # noqa: E501


In [ ]:

from __future__ import annotations

import random
import re
from dataclasses import dataclass, field
from typing import Optional

# ---------------------------------------------------------------------------
# Reading a question (typed, spoken, or one line of an OCR'd page)
# ---------------------------------------------------------------------------

_WORDNUM = {"zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7,
            "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12, "thirteen": 13,
            "fourteen": 14, "fifteen": 15, "sixteen": 16, "seventeen": 17, "eighteen": 18,
            "nineteen": 19, "twenty": 20, "thirty": 30, "forty": 40, "fifty": 50, "sixty": 60,
            "seventy": 70, "eighty": 80, "ninety": 90}
_TENS_W = ("twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety")
_ONES_W = ("one", "two", "three", "four", "five", "six", "seven", "eight", "nine")


def _words_to_digits(t: str) -> str:
    # "twenty-five" / "twenty five" -> 25 ; then single words -> digits
    def tens_ones(m):
        return str(_WORDNUM[m.group(1)] + _WORDNUM[m.group(2)])
    t = re.sub(r"\b(" + "|".join(_TENS_W) + r")[\s-](" + "|".join(_ONES_W) + r")\b", tens_ones, t)
    return re.sub(r"\b(" + "|".join(_WORDNUM) + r")\b", lambda m: str(_WORDNUM[m.group(1)]), t)


SHAPES = {  # name: (sides, corners)
    "triangle": (3, 3), "square": (4, 4), "rectangle": (4, 4), "oblong": (4, 4), "rhombus": (4, 4),
    "kite": (4, 4), "trapezium": (4, 4), "pentagon": (5, 5), "hexagon": (6, 6), "heptagon": (7, 7),
    "octagon": (8, 8), "nonagon": (9, 9), "decagon": (10, 10), "circle": (0, 0), "oval": (0, 0)}
SHAPE_HINTS = {
    "triangle": "a slice of pizza or a tent", "square": "a window or a cracker", "rectangle": "a door or a book",
    "pentagon": "the outline of a house with a pointy roof", "hexagon": "one cell of a bee's honeycomb",
    "octagon": "a STOP sign", "circle": "a coin or a plate", "oval": "an egg"}
FACTS = [  # (regex, answer, kid-friendly hint)
    (r"minutes (?:are )?in (?:an|one|1) hour", 60, "The long hand goes all the way round the clock once. Count by 5s at each number: 5, 10, 15..."),
    (r"minutes (?:are )?in half (?:an|a|1) hour", 30, "Half an hour is the long hand going halfway round - from 12 down to 6. Count by 5s to the 6."),
    (r"minutes (?:are )?in (?:a )?quarter (?:of )?(?:an|a|1) hour", 15, "A quarter hour is the long hand going from 12 to 3. Count by 5s: 5, 10..."),
    (r"days (?:are )?in (?:a|one|1) week", 7, "Say the days with me: Monday, Tuesday... count them on your fingers."),
    (r"months (?:are )?in (?:a|one|1) year", 12, "Start with January and count the months on your fingers."),
    (r"hours (?:are )?in (?:a|one|1) day", 24, "The short hand goes round the clock twice in one day. How many numbers on a clock, times two?"),
    (r"cents (?:are )?in (?:a|one|1) dollar|cents make (?:a|one|1|\$1) dollar", 100, "Think of 10c coins: how many 10c coins make a whole dollar?"),
]


@dataclass
class Problem:
    kind: str                 # add, sub, mul, div, missing, more_less, step_one, digit, build,
                              # round, double, half, quarter, shape, fact, oddeven, multi_add
    text: str                 # the question, tidied, as shown to the kid
    answer: object            # int, or "odd"/"even"
    a: int = 0
    b: int = 0
    c: int = 0
    terms: list = field(default_factory=list)
    extra: dict = field(default_factory=dict)

    @property
    def skill(self) -> str:
        if self.extra.get("money"):
            return "Money"
        if self.extra.get("unit") and self.kind in ("add", "sub"):
            return "Measurement"
        return {"add": "Adding", "multi_add": "Adding", "sub": "Taking away", "mul": "Multiplying (groups of)",
                "div": "Sharing and dividing", "missing": "Missing numbers", "more_less": "10 more / 10 less",
                "step_one": "Counting on and back", "digit": "Place value", "build": "Place value",
                "round": "Rounding", "double": "Doubles and halves", "half": "Doubles and halves",
                "quarter": "Halves and quarters", "shape": "Shapes", "fact": "Time and measurement facts",
                "oddeven": "Odd and even numbers", "money": "Money", "compare": "Comparing numbers",
                "tens": "Place value", "clock": "Telling time", "shape_name": "Shapes",
                "fraction": "Fractions", "convert": "Measurement (changing units)",
                "nth_term": "Number patterns", "two_digits": "Place value",
                "letters": "Patterns (letters)", "weekday": "Days of the week", "before_count": "Counting and order",
                "limit": "Adding and comparing", "word_pick": "Word reasoning", "word_open": "Word reasoning"}.get(self.kind, "Number")


def _ints(t: str) -> list[int]:
    return [int(x) for x in re.findall(r"(?<![\d.])\d+(?!\d|\.\d)", t)]


ADD_WORDS = ("more", "altogether", "in total", "total", "both", "in all", "together", "gets", "got on", "add",
             "plus", "sum", "joined", "came", "arrived", "buys", "bought", "found", "received", "gives her",
             "gives him", "gave her", "gave him", "gives them", "how many pages", "picked")
SUB_WORDS = ("left", "away", "spent", "spend", "gave away", "gave", "ate", "eat", "eats", "lost", "fewer",
             "less", "remain", "take away", "minus", "difference", "popped", "broke", "sold", "change",
             "got off", "off", "flew", "fell", "used", "took", "gave to", "gives to", "now has")


def parse_question(raw: str) -> Optional[Problem]:
    """Python reads the question. Returns None when unsure - never guesses."""
    if not raw or len(raw) > 300:
        return None
    text = re.sub(r"\s+", " ", raw).strip()
    low = text.lower()
    # --- word / reasoning questions that aren't sums (checked BEFORE numbers are read) ---
    wp = _word_problem_kinds(text, low)
    if wp is not None:
        return wp
    # "18 one dollar coins" / "five cent coins": the coin's value is a name, not a quantity
    low = re.sub(r"\b(?:\d+|one|two|five|ten|twenty|fifty)[ -](?:dollar|cent)s? (coins?|notes?)\b", r"\1", low)
    # "this one", "which one", "that one" are pronouns, not the number 1
    low = re.sub(r"\b(this|that|which|each|every|any|another|no|the other|the next|the last) one\b", r"\1 thing", low)
    # numbers inside quotes are names ("Star Trek 7"), not quantities
    low = re.sub(r"[\"“”][^\"“”]{0,40}[\"“”]", " ", low)
    t = _words_to_digits(low).replace("×", "x").replace("÷", "/").replace("−", "-").replace("–", "-")
    t = re.sub(r"(?<=\d),(?=\d{3}\b)", "", t)                    # 1,000 -> 1000
    nums = _ints(t)
    # a spare article-like "one" ("one cricket team scored 56...", "one car which weighs 3")
    # is not a quantity when the real numbers are elsewhere
    t_alt = _words_to_digits(re.sub(r"\bone\b(?= [a-z])", "a", low))
    t_alt = t_alt.replace("×", "x").replace("÷", "/").replace("−", "-").replace("–", "-")
    if len(_ints(t_alt)) >= 2 and len(_ints(t_alt)) < len(nums):
        t, nums = t_alt, _ints(t_alt)

    # --- "Count by 2s: 2, 4, ... What is the 10th one?" (NOT "what comes next") ---
    m = re.search(r"(\d+)(?:st|nd|rd|th) (?:one|1|number|term|step)\b", t)
    if m:
        seq = [int(x) for x in re.findall(r"(?<![\d.])\d+(?!\d|\.\d)", t.split(m.group(0))[0])]
        n = int(m.group(1))
        by = re.search(r"by (\d+)s", t)
        step = int(by.group(1)) if by else (seq[-1] - seq[-2] if len(seq) >= 2 else None)
        if by and seq and seq[0] == step:
            seq = seq[1:]                              # "count by 2s: 2, 4" - the first 2 is the step
        if step and seq and n >= 1 and (len(seq) < 2 or all(seq[i + 1] - seq[i] == step for i in range(len(seq) - 1))):
            first = seq[0]
            return Problem("nth_term", text, first + (n - 1) * step, a=first, b=step, c=n)
        return None

    # --- "How many tens and how many ones are in 97?" (two answers) ---
    m = re.search(r"how many (tens|hundreds) and how many (ones|tens) (?:are )?in (\d+)", t)
    if m:
        n = int(m.group(3))
        hi, lo = m.group(1), m.group(2)
        pv = {"ones": 1, "tens": 10, "hundreds": 100}
        x, y = (n // pv[hi]) % 10, (n // pv[lo]) % 10
        return Problem("two_digits", text, f"{x} {hi} and {y} {lo}", a=x, b=y, c=n, extra={"hi": hi, "lo": lo})

    # --- fractions of a whole: "pizza cut into 8 slices, I eat 3, what fraction is left?" ---
    if "fraction" in t:
        m = re.search(r"(?:cut|split|shared|divided) into (\d+) (?:equal )?(?:slices|pieces|parts|bits)", t)
        if m and len(nums) == 2:
            whole = int(m.group(1)); other = [n for n in nums if n != whole] or [whole]
            k = other[0]
            if 0 < k <= whole:
                left = re.search(r"\b(left|remain|not eaten|still)\b", t) is not None
                top = whole - k if left else k
                return Problem("fraction", text, f"{top}/{whole}", a=top, b=whole,
                               extra={"took": k, "left": left})
        return None                                   # any other fraction question: don't guess
    # --- money with dollars: change, totals, differences (all worked in cents) ---
    if "$" in t or re.search(r"\bdollars?\b", t) or \
            (re.search(r"\d+\s*(?:c|cents?)\b", t) and re.search(r"\b(change|left|remain|spend|spent|buys?|bought|costs?|more|less|difference|cheaper)\b", t)):
        mp = _money_problem(text, t)
        if mp is not None:
            return mp
        if "$" in t and not re.search(r"how many cents", t):
            return None                    # unclear money question: don't guess
    # --- measurement: cm, m, kg, g, L, mL ---
    mm = _measure_problem(text, t, nums)
    if mm is not None:
        return mm
    # units, clocks and anything else unit-ish need their own rules - never a plain sum
    if re.search(r"o'?clock|half past|quarter (past|to)|:\d\d|\b(cm|mm|km|metres?|meters?|m|kg|grams?|g|litres?|liters?|l|ml)\b", t):
        m = re.search(r"(\d+) hours? (after|later than|before|earlier than) (\d+)(?: ?o'?clock)?", t)
        if m and len(nums) == 2 and 1 <= int(m.group(3)) <= 12:
            k, start = int(m.group(1)), int(m.group(3))
            end = (start + k - 1) % 12 + 1 if m.group(2) in ("after", "later than") else (start - k - 1) % 12 + 1
            return Problem("clock", text, end, a=start, b=k, extra={"dir": "after" if m.group(2) in ("after", "later than") else "before"})
        return None

    # --- "which shape has 3 corners?" ---
    m = re.search(r"which shape has (\d+) (sides|corners|vertices|points)", t)
    if m:
        n = int(m.group(1))
        names = [name for name, (sd, cn) in SHAPES.items() if sd == n and name not in ("rectangle", "oblong", "rhombus", "kite", "trapezium")]
        if n in (3, 5, 6, 8):
            return Problem("shape_name", text, names[0], a=n, extra={"what": m.group(2), "ok": [x for x, (sd, _) in SHAPES.items() if sd == n]})
        return None

    # --- shapes and facts (no arithmetic) ---
    for name, (sides, corners) in SHAPES.items():
        if re.search(r"\b" + name + r"s?\b", t) and re.search(r"\b(sides?|corners?|vertices|edges?|points?)\b", t):
            what = "corners" if re.search(r"corners?|vertices|points?", t) else "sides"
            n = corners if what == "corners" else sides
            if name in ("circle", "oval"):
                return None                                      # "0 sides" is a trick question - leave to a grown-up
            return Problem("shape", text, n, extra={"shape": name, "what": what})
    for rx, ans, hint in FACTS:
        if re.search(rx, t) and not nums:
            return Problem("fact", text, ans, extra={"hint": hint})
    m = re.search(r"\b(?:is )?(\d+) (?:an )?(odd|even)(?: or (?:an )?(?:odd|even))?(?: number)?\??$", t) or \
        re.search(r"is (\d+) (?:odd or even|even or odd)", t)
    if m:
        n = int(m.group(1))
        return Problem("oddeven", text, "even" if n % 2 == 0 else "odd", a=n)

    # --- money in coins/notes: "$1 coin and a 50c coin, how many cents" ---
    if re.search(r"\$\s*\d|\d+\s*c\b|cents?", t) and re.search(r"how many cents|how much|total|altogether|both", t) \
            and not re.search(r"change|left|spent|spend|cost.*pay|pay", t):
        parts = [int(d) * 100 for d in re.findall(r"\$\s*(\d+)(?!\.)", t)] + \
                [int(c) for c in re.findall(r"(\d+)\s*(?:c|cents?)\b", t)]
        if 2 <= len(parts) <= 4 and len(parts) == len(nums):
            return Problem("money", text, sum(parts), terms=parts)

    # --- equations with a missing number: 7 + ? = 15, ? - 6 = 10, 3 x _ = 12 ---
    blank = r"(\?|_+|□|\[\s*\]|box)"
    m = re.fullmatch(r"\s*(?:what (?:is|goes in) (?:the )?(?:missing number|box)[:,]?\s*)?(\d+|" + blank[1:-1] + r")\s*([+\-x*/])\s*(\d+|" + blank[1:-1] + r")\s*=\s*(\d+|" + blank[1:-1] + r")\s*\??\s*", t)
    if m:
        x, op, y, z = m.group(1), m.group(2), m.group(3), m.group(4)
        isn = lambda s: s.isdigit()
        op = "*" if op == "x" else op
        if isn(x) and isn(y) and not isn(z):                         # 34 + 48 = ?
            return _binary(text, int(x), op, int(y))
        if [isn(x), isn(y), isn(z)].count(False) == 1:
            X, Y, Z = (int(s) if isn(s) else None for s in (x, y, z))
            ans = None
            if op == "+":
                ans = Z - Y if X is None else Z - X
            elif op == "-":
                ans = Z + Y if X is None else X - Z
            elif op == "*":
                k = Y if X is None else X
                ans = Z // k if k and Z % k == 0 else None
            elif op == "/":
                ans = Z * Y if X is None else (X // Z if Z and X % Z == 0 else None)
            if ans is not None and ans >= 0:
                return Problem("missing", text, ans, a=X if X is not None else -1, b=Y if Y is not None else -1,
                               c=Z, extra={"op": op, "slot": 0 if X is None else 1})
        return None

    # --- bare expressions: "27 + 15", "what is 6 x 4", "4 + 4 + 4" ---
    core = re.sub(r"^(what is|what's|whats|work out|calculate|solve|find|can you do)\s+", "", t).strip(" ?=.!")
    m = re.fullmatch(r"(\d+)\s*([+\-x*/])\s*(\d+)", core)
    if m:
        op = "*" if m.group(2) == "x" else m.group(2)
        return _binary(text, int(m.group(1)), op, int(m.group(3)))
    m = re.fullmatch(r"\d+(?:\s*\+\s*\d+){2,5}", core)
    if m:
        terms = _ints(core)
        return Problem("multi_add", text, sum(terms), terms=terms)

    # --- "sum of 34 and 21", "subtract 15 from 40", "take 8 away from 20", "12 plus 7" ---
    m = re.search(r"(?:sum|total) of (\d+) and (\d+)", t)
    if m:
        return _binary(text, int(m.group(1)), "+", int(m.group(2)))
    m = re.search(r"difference between (\d+) and (\d+)", t)
    if m:
        x, y = int(m.group(1)), int(m.group(2))
        return Problem("sub", text, abs(x - y), a=max(x, y), b=min(x, y), extra={"compare": True})
    m = re.search(r"subtract (\d+) from (\d+)", t) or re.search(r"take (?:away )?(\d+) (?:away )?from (\d+)", t)
    if m:
        return _binary(text, int(m.group(2)), "-", int(m.group(1)))
    words_core = re.sub(r"\b(plus|add|and)\b", "+", core)
    words_core = re.sub(r"\b(minus|take away|subtract)\b", "-", words_core)
    words_core = re.sub(r"\b(times|multiplied by|lots of)\b", "x", words_core)
    words_core = re.sub(r"\b(divided by|shared by|shared between)\b", "/", words_core)
    m = re.fullmatch(r"(\d+)\s*([+\-x/])\s*(\d+)", words_core.strip())
    if m and words_core != core:
        return _binary(text, int(m.group(1)), "*" if m.group(2) == "x" else m.group(2), int(m.group(3)))
    m = re.fullmatch(r"\d+(?:\s*\+\s*\d+){2,5}", words_core.strip())
    if m and words_core != core:
        terms = _ints(words_core)
        return Problem("multi_add", text, sum(terms), terms=terms)

    # --- bigger / smaller ---
    m = re.search(r"which (?:number )?is (bigger|larger|greater|more|smaller|less|lesser)[,:]? (\d+) or (\d+)", t)
    if m and int(m.group(2)) != int(m.group(3)):
        x, y = int(m.group(2)), int(m.group(3))
        want_big = m.group(1) in ("bigger", "larger", "greater", "more")
        return Problem("compare", text, max(x, y) if want_big else min(x, y), a=x, b=y, extra={"big": want_big})

    # --- "I'm thinking of a number..." ---
    m = re.search(r"thinking of a number.*?(?:if|when) i (add|take away|subtract|double) (?:(\d+) )?(?:to it )?(?:i get|it makes|the answer is) (\d+)", t)
    if m:
        op, k, res = m.group(1), m.group(2), int(m.group(3))
        if op == "add" and k and res >= int(k):
            return Problem("missing", text, res - int(k), a=-1, b=int(k), c=res, extra={"op": "+", "slot": 0})
        if op in ("take away", "subtract") and k:
            return Problem("missing", text, res + int(k), a=-1, b=int(k), c=res, extra={"op": "-", "slot": 0})
        if op == "double" and res % 2 == 0:
            return Problem("half", text, res // 2, a=res)

    # --- groups described as "each X has N": legs, eggs in boxes, days in weeks ---
    legs = {"dog": 4, "cat": 4, "cow": 4, "horse": 4, "sheep": 4, "pig": 4, "lion": 4, "tiger": 4, "kangaroo": 2,
            "bird": 2, "chicken": 2, "duck": 2, "person": 2, "people": 2, "child": 2, "children": 2, "kid": 2,
            "spider": 8, "octopus": 8, "ant": 6, "bee": 6, "insect": 6, "beetle": 6, "butterfly": 6}
    m = re.search(r"how many legs (?:do |does |on |have |are on )?(\d+) ([a-z]+?)(?:es|s)?\b", t)
    if m and len(nums) == 1:
        animal = m.group(2)
        if animal in legs:
            per = legs[animal]
            return Problem("mul", text, int(m.group(1)) * per, a=int(m.group(1)), b=per,
                           extra={"story": True, "unit": "legs", "known": f"each {animal} has {per} legs"})
    m = re.search(r"how many (days|minutes|months|hours) (?:are )?in (\d+) (weeks|hours|years|days)", t)
    if m and len(nums) == 1:
        per = {("days", "weeks"): 7, ("minutes", "hours"): 60, ("months", "years"): 12, ("hours", "days"): 24}.get((m.group(1), m.group(3)))
        if per:
            return Problem("mul", text, int(m.group(2)) * per, a=int(m.group(2)), b=per,
                           extra={"story": True, "known": f"there are {per} {m.group(1)} in one {m.group(3)[:-1]}"})
    if len(nums) == 2 and re.search(r"\b(?:a|an|one|each|every)\b[a-z ]{0,25}?\b(?:has|have|holds|hold|costs?|contains?|makes?|is)\s+(\d+)", t) \
            and re.search(r"how many .*?\b(?:on|in|for|with)\s+(\d+)\b", t):
        per = int(re.search(r"\b(?:a|an|one|each|every)\b[a-z ]{0,25}?\b(?:has|have|holds|hold|costs?|contains?|makes?|is)\s+(\d+)", t).group(1))
        count = int(re.search(r"how many .*?\b(?:on|in|for|with)\s+(\d+)\b", t).group(1))
        return Problem("mul", text, per * count, a=count, b=per, extra={"story": True})
    m = re.search(r"(\d+) (\w+) in (?:a|one|each) (\w+)\.? how many \2 (?:are )?in (\d+) \3", t)
    if m:
        return Problem("mul", text, int(m.group(1)) * int(m.group(4)), a=int(m.group(4)), b=int(m.group(1)),
                       extra={"story": True})

    # --- coins: how many 5c coins make 20c; change from $1 ---
    m = re.search(r"how many (\d+)\s*c(?:ents?)? coins (?:make|in|are in|do you need for|for) (\$?\d+)\s*(c|cents?)?", t)
    if m:
        coin = int(m.group(1)); target = m.group(2)
        total = int(target[1:]) * 100 if target.startswith("$") else int(target)
        if coin and total % coin == 0:
            return Problem("div", text, total // coin, a=total, b=coin, extra={"story": True, "share": False, "coins": True})
    m = re.search(r"change (?:from|out of) (?:a )?\$(\d+)(?: note| coin)?.*?(?:spend|spent|cost|costs|buy[a-z ]* for|pay for[a-z ]*?) (\d+)\s*c", t)
    if m:
        paid, cost = int(m.group(1)) * 100, int(m.group(2))
        if cost <= paid:
            return Problem("sub", text, paid - cost, a=paid, b=cost, extra={"story": True, "money": unit})

    # --- tens ---
    m = re.fullmatch(r"(\d+) tens", core)
    if m:
        return Problem("tens", text, 10 * int(m.group(1)), a=int(m.group(1)))
    m = re.search(r"how many (?:10s|tens) (?:are )?(?:there )?in (\d+)", t)
    if m and int(m.group(1)) % 10 == 0 and int(m.group(1)) < 1000:
        n = int(m.group(1))
        return Problem("div", text, n // 10, a=n, b=10, extra={"share": False})

    # --- number words: 10 more/less than, after/before, double, half ---
    m = re.search(r"(\d+) (more|less|fewer) than (\d+)", t)
    if m:
        k, base = int(m.group(1)), int(m.group(3))
        ans = base + k if m.group(2) == "more" else base - k
        if ans >= 0:
            return Problem("more_less", text, ans, a=base, b=k, extra={"dir": "more" if m.group(2) == "more" else "less"})
    m = re.search(r"(?:comes |number |is |one )?(after|before) (\d+)", t)
    if m and len(nums) == 1:
        n = int(m.group(2))
        return Problem("step_one", text, n + 1 if m.group(1) == "after" else n - 1, a=n, extra={"dir": m.group(1)})
    m = re.search(r"double (\d+)", t)
    if m and len(nums) == 1:
        return Problem("double", text, 2 * int(m.group(1)), a=int(m.group(1)))
    m = re.search(r"half (?:of )?(\d+)", t)
    if m and len(nums) == 1 and int(m.group(1)) % 2 == 0:
        return Problem("half", text, int(m.group(1)) // 2, a=int(m.group(1)))
    m = re.search(r"(?:a )?quarter (?:of )?(\d+)", t)
    if m and len(nums) == 1 and int(m.group(1)) % 4 == 0:
        return Problem("quarter", text, int(m.group(1)) // 4, a=int(m.group(1)))

    # --- place value ---
    m = re.search(r"how many (ones|tens|hundreds) (?:are )?(?:there )?in (?:the number )?(\d+)", t) or \
        re.search(r"what (?:digit )?is in the (ones|tens|hundreds) place (?:of|in) (\d+)", t)
    if m:
        n, place = int(m.group(2)), m.group(1)
        if n < 100 or place == "hundreds" or "place" in t:    # "tens in 347" is ambiguous (4 or 34) - skip
            pv = {"ones": 1, "tens": 10, "hundreds": 100}[place]
            return Problem("digit", text, (n // pv) % 10, a=n, extra={"place": place})
    m = re.search(r"(\d+) hundreds?,? (\d+) tens?,? (?:and )?(\d+) ones?", t)
    if m:
        h, te, o = (int(x) for x in m.groups())
        if te < 10 and o < 10:
            return Problem("build", text, 100 * h + 10 * te + o, terms=[h, te, o])
    m = re.search(r"(\d+) tens?,? (?:and )?(\d+) ones?", t)
    if m and len(nums) == 2:
        te, o = int(m.group(1)), int(m.group(2))
        if o < 10:
            return Problem("build", text, 10 * te + o, terms=[0, te, o])

    # --- rounding ---
    m = re.search(r"round (\d+) to the nearest (ten|10|hundred|100)\b", t) or \
        re.search(r"(\d+) rounded to the nearest (ten|10|hundred|100)\b", t)
    if m:
        n, unit = int(m.group(1)), 10 if m.group(2) in ("ten", "10") else 100
        return Problem("round", text, (n + unit // 2) // unit * unit, a=n, b=unit)

    # --- groups / sharing ---
    m = re.search(r"(\d+) (?:groups|rows|lots|bags|boxes|packs|packets|plates|teams|jars|baskets|sets|tables|cars|trays|vases|bunches|piles|stacks|columns) (?:of|with|have|has|each have|each has|holding|contain|containing) (\d+)", t)
    if m and len(nums) == 2:
        return Problem("mul", text, int(m.group(1)) * int(m.group(2)), a=int(m.group(1)), b=int(m.group(2)),
                       extra={"story": True})
    m = re.search(r"(\d+) (\w+) (?:and|with) (\d+) \w+ (?:on|in|at|for) (?:each|every) \w+", t)
    if m and len(nums) == 2:
        return Problem("mul", text, int(m.group(1)) * int(m.group(3)), a=int(m.group(1)), b=int(m.group(3)),
                       extra={"story": True})
    m = re.search(r"(\d+) \w+ (?:with|have|has|holds?|contains?) (\d+) \w+ (?:in )?each", t)
    if m and len(nums) == 2:
        return Problem("mul", text, int(m.group(1)) * int(m.group(2)), a=int(m.group(1)), b=int(m.group(2)),
                       extra={"story": True})
    if re.search(r"shar|equal|each|split|between|among", t) and len(nums) == 2 and \
            re.search(r"how many (?:does |do |will )?(?:each|in each)|each get|each\?|how many each|in each group", t):
        total, groups = (nums[0], nums[1]) if nums[0] >= nums[1] else (nums[1], nums[0])
        if groups and total % groups == 0:
            return Problem("div", text, total // groups, a=total, b=groups, extra={"story": True, "share": True})
    m = re.search(r"(\d+) .*?(?:into|in) groups of (\d+)", t)
    if m and len(nums) == 2 and re.search(r"how many groups", t):
        total, size = int(m.group(1)), int(m.group(2))
        if size and total % size == 0:
            return Problem("div", text, total // size, a=total, b=size, extra={"story": True, "share": False})

    # --- two-number word problems: add or take away ---
    if re.search(r"\b(each|every|per)\b", t):
        return None                      # groups/sharing wording we didn't recognise: don't guess + or -
    if len(nums) == 2 and re.search(r"how many|how much|what is the total|altogether|left", t):
        a, b = nums
        is_sub = any(re.search(r"\b" + w + r"\b", t) for w in SUB_WORDS)
        is_add = any(re.search(r"\b" + w + r"\b", t) for w in ADD_WORDS)
        # "gives her 17 more" is adding even though "gives"/"gave" can mean taking away;
        # "gave 21 to her brother" is taking away. Anything still mixed: don't guess.
        if re.search(r"\b(gives?|gave) (her|him|them|me|us) \d+", t):
            is_sub = any(re.search(r"\b" + w + r"\b", t) for w in SUB_WORDS if w not in ("gave", "gave away"))
        if "how many more" in t or "difference" in t:
            big, small = max(a, b), min(a, b)
            return Problem("sub", text, big - small, a=big, b=small, extra={"story": True, "compare": True})
        if is_sub and not is_add and a >= b:
            return Problem("sub", text, a - b, a=a, b=b, extra={"story": True})
        if is_add and not is_sub:
            return Problem("add", text, a + b, a=a, b=b, extra={"story": True})
    return None


UNIT_WORDS = {"cm": "cm", "centimetre": "cm", "centimetres": "cm", "centimeter": "cm", "centimeters": "cm",
              "m": "m", "metre": "m", "metres": "m", "meter": "m", "meters": "m",
              "mm": "mm", "km": "km", "kg": "kg", "kilogram": "kg", "kilograms": "kg", "kilo": "kg", "kilos": "kg",
              "g": "g", "gram": "g", "grams": "g", "l": "L", "litre": "L", "litres": "L", "liter": "L",
              "liters": "L", "ml": "mL", "millilitre": "mL", "millilitres": "mL", "milliliter": "mL", "milliliters": "mL"}
CONVERT = {("cm", "m"): 100, ("mm", "cm"): 10, ("m", "km"): 1000, ("g", "kg"): 1000, ("mL", "L"): 1000}
SUB_MEASURE = r"\b(cut off|cuts off|cut|used|uses|drank|drinks|poured out|pours out|ate|eats|lost|left|remain|now|gave away|took)\b"
ADD_MEASURE = r"\b(total|altogether|both|together|in all|combined|added|adds|joined|end to end|another|more)\b"
CMP_MEASURE = r"how much (longer|shorter|taller|heavier|lighter|more|less|further|wider)|difference"


def _measure_problem(text: str, t: str, nums: list) -> Optional["Problem"]:
    # conversions: "how many cm in 1 metre", "how many grams in 2 kg"
    m = re.search(r"how many (\w+) (?:are )?(?:there )?in (?:a |an |one |(\d+) )?(\w+)", t)
    if m and m.group(1) in UNIT_WORDS and m.group(3) in UNIT_WORDS:
        small, big = UNIT_WORDS[m.group(1)], UNIT_WORDS[m.group(3)]
        k = int(m.group(2) or 1)
        if (small, big) in CONVERT and len(nums) <= 1:
            per = CONVERT[(small, big)]
            return Problem("convert", text, per * k, a=k, b=per, extra={"small": small, "big": big, "unit": small})
    pairs = [(int(v), UNIT_WORDS[u]) for v, u in re.findall(r"(\d+)\s*(cm|mm|km|kg|ml|m|g|l|centimetres?|centimeters?|metres?|meters?|"
                                                              r"kilograms?|kilos?|grams?|litres?|liters?|millilitres?|milliliters?)\b", t)]
    if not pairs:
        return None
    units = {u for _, u in pairs}
    if len(pairs) != 2 or len(units) != 1 or len(nums) != 2:
        return None                        # mixed units / extra numbers: leave it to a grown-up or the checked path
    unit = units.pop()
    (a, _), (b, _) = pairs
    if re.search(CMP_MEASURE, t):
        big, small = max(a, b), min(a, b)
        return Problem("sub", text, big - small, a=big, b=small, extra={"story": True, "compare": True, "unit": unit})
    is_sub, is_add = bool(re.search(SUB_MEASURE, t)), bool(re.search(ADD_MEASURE, t))
    if is_sub and not is_add and a >= b:
        return Problem("sub", text, a - b, a=a, b=b, extra={"story": True, "unit": unit})
    if is_add and not is_sub:
        return Problem("add", text, a + b, a=a, b=b, extra={"story": True, "unit": unit})
    return None


def _cents(tok: str) -> Optional[int]:
    """'$2' -> 200, '$1.50' -> 150, '50c' -> 50."""
    tok = tok.strip()
    m = re.fullmatch(r"\$\s*(\d+)(?:\.(\d{1,2}))?", tok)
    if m:
        return int(m.group(1)) * 100 + (int((m.group(2) or "0").ljust(2, "0")) if m.group(2) else 0)
    m = re.fullmatch(r"(\d+)\s*(?:c|cents?)", tok)
    return int(m.group(1)) if m else None


def _money_problem(text: str, t: str) -> Optional["Problem"]:
    amounts = [c for c in (_cents(x) for x in re.findall(r"\$\s*\d+(?:\.\d{1,2})?|\d+\s*(?:c|cents?)\b", t)) if c is not None]
    t2 = re.sub(r"(\d+) dollars?", lambda mm: "$" + mm.group(1), t)
    if t2 != t:
        amounts = [c for c in (_cents(x) for x in re.findall(r"\$\s*\d+(?:\.\d{1,2})?|\d+\s*(?:c|cents?)\b", t2)) if c is not None]
    if len(amounts) != 2:
        return None
    x, y = amounts
    # whole dollars are taught in dollars ("$5 - $3"), anything with cents in cents
    unit = "dollars" if x % 100 == 0 and y % 100 == 0 else "cents"
    if unit == "dollars":
        x, y = x // 100, y // 100
    if re.search(r"\bchange\b", t):
        paid, cost = max(x, y), min(x, y)
        return Problem("sub", text, paid - cost, a=paid, b=cost, extra={"story": True, "money": unit, "change": True})
    if re.search(r"how much more|how much less|difference|more expensive|cheaper", t):
        return Problem("sub", text, abs(x - y), a=max(x, y), b=min(x, y), extra={"story": True, "money": unit, "compare": True})
    if re.search(r"\b(spend|spent|left|remain|pays? for|bought|buys?)\b", t) and re.search(r"\b(left|remain|now|still)\b", t):
        if x >= y:
            return Problem("sub", text, x - y, a=x, b=y, extra={"story": True, "money": unit})
    if re.search(r"\b(total|altogether|both|together|in all|how much for|how much do .* cost|cost of)\b", t):
        return Problem("add", text, x + y, a=x, b=y, extra={"story": True, "money": unit})
    return None


def fmt_answer(p: "Problem", v=None) -> str:
    """How an answer is SAID: money as $/c, measurements with their unit."""
    v = p.answer if v is None else v
    if not isinstance(v, int):
        return str(v)
    if p.extra.get("money") == "dollars":
        return f"${v}"
    if p.extra.get("money"):
        if v >= 100:
            return f"${v // 100}" + (f".{v % 100:02d}" if v % 100 else "")
        return f"{v}c"
    if p.extra.get("unit"):
        return f"{v} {p.extra['unit']}"
    return str(v)


def _count_start(k: int, answer, unit: str = "") -> str:
    """'3, 6...' to start a skip count - but never a list that contains the answer."""
    shown = [n for n in (k, 2 * k) if n != answer]
    if len(shown) < 2:
        shown = shown[:1]
    return ", ".join(f"{n}{unit}" for n in shown) + "..." if shown else "..."


DAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
ORDINAL_WORDS = {"first": 1, "second": 2, "third": 3, "fourth": 4, "fifth": 5, "sixth": 6, "seventh": 7,
                 "eighth": 8, "ninth": 9, "tenth": 10, "eleventh": 11, "twelfth": 12}
CATEGORIES = {
    "farm animal": {"horse", "cow", "sheep", "pig", "goat", "chicken", "hen", "duck", "donkey", "rooster", "turkey", "lamb", "calf"},
    "fruit": {"apple", "banana", "orange", "pear", "grape", "mango", "peach", "plum", "cherry", "strawberry", "kiwi", "lemon", "watermelon"},
    "vegetable": {"carrot", "pea", "potato", "onion", "broccoli", "cabbage", "lettuce", "pumpkin", "corn", "bean", "tomato"},
    "colour": {"red", "blue", "green", "yellow", "orange", "purple", "pink", "black", "white", "brown", "grey", "gray"},
    "color": {"red", "blue", "green", "yellow", "orange", "purple", "pink", "black", "white", "brown", "grey", "gray"},
    "bird": {"parrot", "eagle", "owl", "duck", "chicken", "emu", "kookaburra", "magpie", "pigeon", "penguin", "swan"},
    "insect": {"bee", "ant", "butterfly", "fly", "beetle", "mosquito", "ladybird", "grasshopper"},
    "shape": {"circle", "square", "triangle", "rectangle", "hexagon", "pentagon", "octagon", "oval"},
    "number": {"one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten"},
}
OPPOSITES = {
    "stupid": {"clever", "smart", "wise", "bright", "intelligent"}, "clever": {"stupid", "silly", "foolish"},
    "hot": {"cold"}, "cold": {"hot", "warm"}, "big": {"small", "little", "tiny"}, "small": {"big", "large", "huge"},
    "happy": {"sad", "unhappy"}, "sad": {"happy", "glad"}, "fast": {"slow"}, "slow": {"fast", "quick"},
    "up": {"down"}, "down": {"up"}, "open": {"closed", "shut"}, "full": {"empty"}, "empty": {"full"},
    "light": {"dark", "heavy"}, "dark": {"light", "bright"}, "old": {"new", "young"}, "young": {"old"},
    "wet": {"dry"}, "dry": {"wet"}, "early": {"late"}, "late": {"early"}, "tall": {"short"}, "long": {"short"},
    "hard": {"soft", "easy"}, "soft": {"hard"}, "loud": {"quiet", "soft"}, "quiet": {"loud", "noisy"},
    "rich": {"poor"}, "poor": {"rich"}, "kind": {"cruel", "mean", "unkind"}, "brave": {"scared", "afraid", "cowardly"},
    "top": {"bottom"}, "push": {"pull"}, "win": {"lose"}, "day": {"night"}, "first": {"last"},
}


def _options(text: str) -> list[str]:
    """Multiple-choice words written in CAPITALS after the question mark."""
    tail = text.split("?", 1)[1] if "?" in text else text
    return [w.lower() for w in re.findall(r"\b[A-Z]{2,}\b", tail)]


def _word_problem_kinds(text: str, low: str) -> Optional["Problem"]:
    # letter patterns: "B, C, D, E, ?" or "N, M, L, K, J, ?"
    m = re.search(r"\b([A-Za-z])(?:\s*,\s*([A-Za-z])\b){2,}\s*,?\s*(?:\?|_+|…)", text)
    if m:
        seq = [c.upper() for c in re.findall(r"\b([A-Za-z])\b", m.group(0))]
        idx = [ord(c) - 65 for c in seq]
        steps = {idx[i + 1] - idx[i] for i in range(len(idx) - 1)}
        if len(seq) >= 3 and len(steps) == 1 and 0 < abs(next(iter(steps))) <= 3:
            st = next(iter(steps))
            nxt = idx[-1] + st
            if 0 <= nxt < 26:
                return Problem("letters", text, chr(65 + nxt), a=idx[-1], b=st, terms=seq)
        return None
    # days of the week: "the day after Tuesday", "two days before Friday"
    m = re.search(r"\b(?:the )?(?:(\w+) )?days? (after|before) (" + "|".join(DAYS) + r")\b", low)
    if m:
        k = {"a": 1, "the": 1, None: 1, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6}.get(m.group(1), None)
        if m.group(1) and m.group(1).isdigit():
            k = int(m.group(1))
        if k is None:
            k = 1
        start = DAYS.index(m.group(3))
        end = (start + (k if m.group(2) == "after" else -k)) % 7
        return Problem("weekday", text, DAYS[end].capitalize(), a=start, b=k, extra={"dir": m.group(2)})
    # "the seventh film ... how many came before this one?"
    m = re.search(r"\b(" + "|".join(ORDINAL_WORDS) + r"|\d+(?:st|nd|rd|th))\b.*\bhow many\b.*\b(before|earlier|came first)\b", low)
    if m:
        w = m.group(1)
        n = ORDINAL_WORDS.get(w) or int(re.match(r"\d+", w).group(0))
        return Problem("before_count", text, n - 1, a=n)
    # weight limit: "limit of 8 tonnes ... weighs three tonnes ... weighs seven tonnes, will it break?"
    if re.search(r"\b(limit|maximum|max|hold up to|can hold)\b", low) and re.search(r"will it (break|hold|sink|be ok)", low):
        d = _words_to_digits(low)
        lim = re.search(r"(?:limit of|maximum of|max of|up to|can hold) (\d+) ?(tonnes?|kg|kilograms?|g)?", d)
        loads = [int(x) for x in re.findall(r"weighs? (\d+)", d)]
        if lim and len(loads) >= 2:
            total, limit = sum(loads), int(lim.group(1))
            too_much = total > limit
            holds_q = re.search(r"will it (hold|be ok)", low) is not None      # "will it hold?" flips yes/no
            word = re.search(r"will it (break|hold|sink|be ok)", low).group(1)
            answer = ("no" if too_much else "yes") if holds_q else ("yes" if too_much else "no")
            return Problem("limit", text, answer,
                           a=total, b=limit, terms=loads, extra={"unit": (lim.group(2) or "").rstrip("s") + "s" if lim.group(2) else "", "word": word})
        return None
    # word puzzles with capital-letter choices
    opts = _options(text)
    if len(opts) >= 3:
        m = re.search(r"which (?:word|one) (?:is|names) (?:a|an) ([a-z ]+?)\?", low)
        if m:
            cat = m.group(1).strip().rstrip("s")
            hits = [o for o in opts if o in CATEGORIES.get(cat, set())]
            if cat in CATEGORIES and len(hits) == 1:
                return Problem("word_pick", text, hits[0].upper(), terms=opts, extra={"cat": cat, "how": "category"})
            return Problem("word_open", text, None, terms=opts, extra={"ask": f"Which one is a {cat}?"})
        m = re.search(r"opposite of ([A-Za-z]+)", text)
        if m:
            target = m.group(1).lower()
            hits = [o for o in opts if o in OPPOSITES.get(target, set())]
            if len(hits) == 1:
                return Problem("word_pick", text, hits[0].upper(), terms=opts, extra={"target": target, "how": "opposite"})
            return Problem("word_open", text, None, terms=opts, extra={"ask": f"Which word means the opposite of {target.upper()}?"})
        if re.search(r"odd one out|doesn'?t belong|does not belong", low):
            return Problem("word_open", text, None, terms=opts, extra={"ask": "Which word is the odd one out?", "odd": True})
    return None


def _binary(text, a, op, b) -> Optional[Problem]:
    if op == "+":
        return Problem("add", text, a + b, a=a, b=b)
    if op == "-":
        return Problem("sub", text, a - b, a=a, b=b) if a >= b else None
    if op == "*":
        return Problem("mul", text, a * b, a=a, b=b)
    if op == "/":
        return Problem("div", text, a // b, a=a, b=b) if b and a % b == 0 else None
    return None


# ---------------------------------------------------------------------------
# Teaching it - every number below is computed, nothing is made up
# ---------------------------------------------------------------------------

def plan_steps(p: Problem) -> list[tuple[str, object]]:
    """Little steps a kid can answer on the way: (question to ask, expected value)."""
    a, b = p.a, p.b
    if p.kind == "sub" and p.extra.get("change"):
        p.extra["strategy"] = "count_up_money"
        return [(f"Shopkeepers count UP to work out change! Start at {fmt_answer(p, b)} (the cost) and count up to "
                 f"{fmt_answer(p, a)} (what was paid). How much did you count up?", p.answer)]
    if p.kind in ("add", "sub") and 0 in (a, b) and not p.extra.get("compare"):
        p.extra["strategy"] = "zero"
        return [("Here's a fun rule: adding or taking away 0 changes NOTHING - zero means none! "
                 "So what's the answer?", p.answer)]
    if p.kind in ("add", "sub") and (b if p.kind == "sub" else min(a, b)) >= 100 and not p.extra.get("compare"):
        # 3-digit: hundreds, then tens, then ones (only the parts that aren't 0)
        start, part_of = (a, b) if p.kind == "sub" else (max(a, b), min(a, b))
        parts = [x for x in (part_of // 100 * 100, part_of // 10 % 10 * 10, part_of % 10) if x]
        names = {0: "hundreds", 1: "tens", 2: "ones"}
        labels = [names[i] for i, x in enumerate((part_of // 100 * 100, part_of // 10 % 10 * 10, part_of % 10)) if x]
        sign = "+" if p.kind == "add" else "-"
        steps, run = [], start
        for i, (x, lab) in enumerate(zip(parts, labels)):
            nxt = run + x if p.kind == "add" else run - x
            verb = "add" if p.kind == "add" else "take away"
            lead = f"Let's split {part_of} into hundreds, tens and ones and {verb} one part at a time. First the {lab}: " if i == 0 \
                else f"Now the {lab}: "
            steps.append((f"{lead}what's {run} {sign} {x}?", nxt))
            run = nxt
        p.extra["strategy"] = "place_parts"
        return steps
    if p.kind == "add":
        big, small = (a, b) if a >= b else (b, a)
        tens, ones = small // 10 * 10, small % 10
        if small >= 10 and ones:
            p.extra["strategy"] = "tens_then_ones"
            return [(f"Let's add the tens first. What's {big} + {tens}?", big + tens),
                    (f"Now add the {ones} ones that are left. What's {big + tens} + {ones}?", p.answer)]
        if small >= 10:
            p.extra["strategy"] = "jump_tens"
            return [(f"Count on in tens from {big} - that's {small // 10} jumps of ten. Where do you land?", p.answer)]
        if big < 10 and big + small > 10:
            p.extra["strategy"] = "make_ten"
            need = 10 - big
            return [(f"Let's make a ten first! How many more does {big} need to make 10?", need),
                    (f"Yes! So {small} splits into {need} and {small - need}. What's 10 + {small - need}?", p.answer)]
        if small == 9 and big >= 10:
            p.extra["strategy"] = "add_ten_take_one"
            return [(f"Here's a trick: adding 9 is like adding 10, then taking 1 away. What's {big} + 10?", big + 10),
                    (f"Now take 1 away: what's {big + 10} - 1?", p.answer)]
        p.extra["strategy"] = "count_on"
        return [(f"Start at {big} and count on {small} more on your fingers. Where do you land?", p.answer)]
    if p.kind == "sub":
        tens, ones = b // 10 * 10, b % 10
        if p.extra.get("compare"):
            p.extra["strategy"] = "count_up"
            return [(f"Start at {b} and count up to {a}. How many jumps is that?", p.answer)]
        if b >= 10 and ones:
            p.extra["strategy"] = "tens_then_ones"
            return [(f"Let's take away the tens first. What's {a} - {tens}?", a - tens),
                    (f"Now take away the {ones} ones. What's {a - tens} - {ones}?", p.answer)]
        if b >= 10:
            p.extra["strategy"] = "jump_tens"
            return [(f"Count back in tens from {a} - that's {b // 10} jumps of ten. Where do you land?", p.answer)]
        if 10 < a < 20 and a % 10 < b:
            p.extra["strategy"] = "back_to_ten"
            first = a % 10
            return [(f"Let's jump back to 10 first. How many do we take away from {a} to get to 10?", first),
                    (f"Yes! We still need to take away {b - first} more. What's 10 - {b - first}?", p.answer)]
        if b == 9 and a >= 10:
            p.extra["strategy"] = "take_ten_add_one"
            return [(f"Here's a trick: taking 9 is like taking 10, then giving 1 back. What's {a} - 10?", a - 10),
                    (f"Now give 1 back: what's {a - 10} + 1?", p.answer)]
        p.extra["strategy"] = "count_back"
        return [(f"Start at {a} and count back {b} on your fingers. Where do you land?", p.answer)]
    if p.kind == "multi_add":
        steps, run = [], p.terms[0]
        for t in p.terms[1:-1]:
            steps.append((f"Add them one at a time. What's {run} + {t}?", run + t))
            run += t
        steps.append((f"And now the last one: what's {run} + {p.terms[-1]}?", p.answer))
        return steps
    if p.kind == "money":
        cents = p.terms
        return [(f"Turn everything into cents first: that's {' and '.join(f'{c}c' for c in cents)}. "
                 f"Now add them up - what do you get?", p.answer)]
    if p.kind == "mul":
        groups, size = (a, b) if p.extra.get("story") else (a, b)
        if groups == 2:
            return [(f"2 groups of {size} is the same as double {size}. What's {size} + {size}?", p.answer)]
        known = (p.extra.get("known", "") + " - so that's ") if p.extra.get("known") else "That's "
        return [(f"{known[0].upper() + known[1:]}{groups} groups of {size}. Let's skip count by {size}s: "
                 f"{_count_start(size, p.answer)} keep going until you've said {groups} numbers. Where do you finish?", p.answer)]
    if p.kind == "div":
        total, k = a, b
        if p.extra.get("share", True):
            return [(f"Let's share {total} into {k} equal groups. Count by {k}s: {_count_start(k, p.answer)} "
                     f"how many jumps until you reach {total}?", p.answer)]
        if p.extra.get("coins"):
            return [(f"Count up in {k}c coins: {_count_start(k, p.answer, 'c')} until you reach {total}c. How many coins did you count?", p.answer)]
        return [(f"How many groups of {k} fit into {total}? Count by {k}s until you reach {total} - count your jumps.", p.answer)]
    if p.kind == "missing":
        op, slot, c = p.extra["op"], p.extra["slot"], p.c
        known = p.b if slot == 0 else p.a
        if op == "+":
            return [(f"What do we add to {known} to make {c}? Start at {known} and count up to {c} - how many jumps?", p.answer)]
        if op == "-" and slot == 0:
            return [(f"Let's work backwards! Something take away {known} leaves {c}. So start at {c} and add the {known} back on. What do you get?", p.answer)]
        if op == "-":
            return [(f"From {known}, how many do we take away to land on {c}? Count back from {known} to {c} - how many jumps?", p.answer)]
        if op == "*":
            return [(f"How many groups of {known} make {c}? Count by {known}s until you reach {c}.", p.answer)]
        return [(f"Let's think about it with times: what number shared into {known} gives {c}?", p.answer)]
    if p.kind == "more_less":
        word = "more" if p.extra["dir"] == "more" else "less"
        if b == 10:
            return [(f"When we add or take 10, only the tens digit changes! {a} has {a // 10 % 10} tens. "
                     f"What's 1 {'more' if word == 'more' else 'less'} ten than that? Then put the number back together.", p.answer)]
        return [(f"Start at {a} and count {'on' if word == 'more' else 'back'} {b}. Where do you land?", p.answer)]
    if p.kind == "step_one":
        return [(f"Count {'up' if p.extra['dir'] == 'after' else 'back'} just one from {a}. What number is that?", p.answer)]
    if p.kind == "digit":
        place = p.extra["place"]
        return [(f"Let's look at {a} digit by digit. The ones are on the right, the tens next door, then the hundreds. "
                 f"Which digit sits in the {place} place?", p.answer)]
    if p.kind == "build":
        h, te, o = p.terms
        if h:
            return [("How many hundreds are there? That digit goes first, in the hundreds place.", h),
                    ("Next door is the tens place. Which digit goes there?", te),
                    ("And the ones digit goes last. Now read the whole number - what is it?", p.answer)]
        return [("How many tens? That digit goes first, in the tens place.", te),
                ("Then the ones digit. Now read the whole number - what is it?", p.answer)]
    if p.kind == "round":
        unit = b
        look = a % 10 if unit == 10 else (a // 10) % 10
        place = "ones" if unit == 10 else "tens"
        return [(f"Here's the rule: look at the {place} digit of {a} - it's {look}. If it's 5 or more we round UP, "
                 f"if it's 4 or less we round DOWN. So which {'ten' if unit == 10 else 'hundred'} do you get?", p.answer)]
    if p.kind == "compare":
        x, y = a, b
        if x // 10 != y // 10:
            return [(f"Compare the tens first: {x} has {x // 10} tens and {y} has {y // 10} tens. "
                     f"So which number is {'bigger' if p.extra['big'] else 'smaller'}?", p.answer)]
        return [(f"They have the same tens, so look at the ones: {x % 10} and {y % 10}. "
                 f"Which number is {'bigger' if p.extra['big'] else 'smaller'}?", p.answer)]
    if p.kind == "double":
        return [(f"Double means two lots of the same. What's {a} + {a}?", p.answer)]
    if p.kind == "half":
        return [(f"Half means sharing into 2 equal groups. What number plus itself makes {a}?", p.answer)]
    if p.kind == "quarter":
        return [(f"A quarter is half of a half! First, what's half of {a}?", a // 2),
                (f"Now halve it again: what's half of {a // 2}?", p.answer)]
    if p.kind == "shape":
        hint = SHAPE_HINTS.get(p.extra["shape"])
        look = f"A {p.extra['shape']} looks like {hint}. " if hint else ""
        return [(f"{look}Imagine tracing around it with your finger and count the {p.extra['what']} as you go. How many?", p.answer)]
    if p.kind == "fact":
        return [(p.extra["hint"], p.answer)]
    if p.kind == "letters":
        way = "forwards" if b > 0 else "backwards"
        jump = abs(b)
        hop = "the very next letter" if jump == 1 else f"jump {jump} letters"
        return [(f"Say the alphabet {way} from {p.terms[-1]}. The pattern goes {way}, so {hop}. Which letter comes next?", p.answer)]
    if p.kind == "weekday":
        return [(f"Let's say the days of the week together: Monday, Tuesday, Wednesday, Thursday, Friday, Saturday, Sunday. "
                 f"Find {DAYS[a].capitalize()}, then go {'forward' if p.extra['dir'] == 'after' else 'back'} "
                 f"{b} day{'s' if b != 1 else ''}. Which day is it?", p.answer)]
    if p.kind == "before_count":
        return [(f"If this one is number {a}, the ones before it are number 1, number 2... up to the number just before {a}. "
                 f"How many is that?", p.answer)]
    if p.kind == "limit":
        unit = p.extra.get("unit") or ""
        return [(f"First, add up everything on it: {' + '.join(str(x) for x in p.terms)} = ?", a),
                (f"Now compare: is that MORE than the limit of {b} {unit}? So, will it {p.extra['word']} - yes or no?", p.answer)]
    if p.kind == "word_pick":
        if p.extra["how"] == "category":
            return [(f"Read each word and ask: is it a {p.extra['cat']}? "
                     f"{', '.join(o.upper() for o in p.terms)}. Which one is it?", p.answer)]
        return [(f"An opposite means the other end - like hot and cold. What's the opposite of {p.extra['target'].upper()}? "
                 f"Look at: {', '.join(o.upper() for o in p.terms)}.", p.answer)]
    if p.kind == "word_open":
        if p.extra.get("odd"):
            return [("Find the group! Say each word and think: which ones belong together - where would you find them? "
                     f"{', '.join(o.upper() for o in p.terms)}. Which one doesn't fit, and why?", None)]
        return [(f"{p.extra.get('ask')} Read each word slowly: {', '.join(o.upper() for o in p.terms)}.", None)]
    if p.kind == "nth_term":
        return [(f"Let's count by {b}s and keep track on your fingers: {a} is number 1, {a + b} is number 2... "
                 f"keep going until you reach number {p.c}. What do you say?", p.answer)]
    if p.kind == "two_digits":
        hi, lo = p.extra["hi"], p.extra["lo"]
        return [(f"Let's look at {p.c} digit by digit. Which digit is in the {hi} place?", a),
                (f"And which digit is in the {lo} place? Then say both: '_ {hi} and _ {lo}'.", p.answer)]
    if p.kind == "convert":
        sm, bg, per = p.extra["small"], p.extra["big"], b
        if a == 1:
            return [(f"Here's a fact worth remembering: think of a {('metre ruler' if bg == 'm' else bg)} - how many {sm} fit in it? "
                     f"(Hint: it's a 1 with zeros after it!)", p.answer)]
        return [(f"First, how many {sm} are in ONE {bg}?", per),
                (f"So for {a} {bg}, that's {a} lots of that. What do you get?", p.answer)]
    if p.kind == "tens":
        return [(f"{a} tens means {a} groups of ten. Count by 10s, {a} times, on your fingers. Where do you finish?", p.answer)]
    if p.kind == "clock":
        way = "forward" if p.extra["dir"] == "after" else "back"
        return [(f"Picture the clock at {a} o'clock. Move the short hour hand {way} {b} {'hour' if b == 1 else 'hours'} - "
                 f"one number for each hour. Which number does it point to now?", p.answer)]
    if p.kind == "shape_name":
        return [(f"Let's think of shapes and count their {p.extra['what']}: a square has 4, a circle has none... "
                 f"which shape has exactly {a}? Try drawing a few!", p.answer)]
    if p.kind == "fraction":
        took, whole = p.extra["took"], b
        if p.extra["left"]:
            return [(f"There were {whole} slices and {took} got eaten. How many slices are left?", a),
                    (f"Yes! So that many out of {whole} slices are left. A fraction is written 'slices left / all the slices'. "
                     f"How do you write it?", p.answer)]
        return [(f"A fraction is written 'the part / all the parts'. Here the part is {a} and there are {whole} parts altogether. "
                 f"How do you write it?", p.answer)]
    if p.kind == "oddeven":
        last = p.a % 10
        return [(f"Just look at the last digit: {last}. Can {last} things be put into pairs with none left over? "
                 f"If yes it's even, if one is left over it's odd.", p.answer)]
    return [("What do you think the answer is?", p.answer)]


# ---------------------------------------------------------------------------
# How GEETA talks: a friendly buddy, not a textbook. Varied, short, warm.
# ---------------------------------------------------------------------------

PRAISE = ["Yes!! 🎉", "You got it! ⭐", "Woohoo, that's it! 🙌", "Yesss, nailed it! 🌟", "That's right - high five! ✋",
          "Spot on! 🎯", "Boom, you did it! 💥"]
STEP_PRAISE = ["Yes! {n} is right.", "Ooh, {n} - perfect.", "Yep, {n}! Good thinking.", "{n} - exactly right!",
               "Nice one, {n} is spot on."]
STUCK_COMFORT = ["That's totally okay - tricky ones make our brains grow! 🌱", "No worries, we'll do it together.",
                 "Hmm, it IS a tricky one. Let's make it smaller.", "That's fine! Every maths whiz gets stuck sometimes.",
                 "Okay, let's take it nice and slow."]
TRY_AGAIN = ["Not quite - but good try!", "Ooh, close-ish! Let's check it.", "Hmm, not that one. Let's look again.",
             "Nearly! Let's have another look together."]
NO_SPOILER = ["Ha, nice try! 😄 I'm not allowed to spill it - but I'll help you get it, promise.",
              "I could... but then YOU wouldn't get the fun of cracking it! Here's a clue instead:",
              "No spoilers from me! 🤐 But I've got a good clue for you:",
              "If I tell you, your brain misses out on the workout! 💪 Here's a clue:"]
FEELINGS = {
    "tired": ["You sound tired - that's okay. Let's just do one tiny bit, then you can rest. 😴",
              "Tired brains still do great maths! Just one little step:"],
    "bored": ["Bored? Let's make it a game then - can you beat me to it? 😄", "Okay, speed round! Quick:"],
    "hard": ["It feels hard because you're learning something new - that's your brain growing! 🌱",
             "Hard ones are the best ones to beat. Let's chop it into a smaller piece."],
    "sad": ["Aw, I'm sorry you feel sad. 💛 We can go slowly. Want to try just a tiny bit with me?"],
    "cant": ["You CAN - you just can't YET. Let's do the first tiny bit together.",
             "I believe in you! Let's start really small."],
}


def pick(xs):
    return random.choice(xs)


def feeling_of(t: str) -> Optional[str]:
    t = t.lower()
    if re.search(r"\b(tired|sleepy|yawn)\b", t):
        return "tired"
    if re.search(r"\b(bored|boring)\b", t):
        return "bored"
    if re.search(r"\b(sad|upset|crying|cry)\b", t):
        return "sad"
    if re.search(r"\b(too hard|so hard|hard|difficult|tricky)\b", t):
        return "hard"
    if re.search(r"\b(can'?t do|cannot do|i can'?t|i'?m bad at|i'?m dumb|i'?m stupid|give up)\b", t):
        return "cant"
    return None


ASK_ANSWER = re.compile(r"(just )?(tell|give|say) me (the )?answer|what'?s the answer|what is the answer|"
                        r"you do it|you tell me|just say it", re.I)
STUCK = re.compile(r"\b(stuck|don'?t know|dont know|do not know|idk|no idea|help|hint|not sure|confused|"
                   r"don'?t get it|how do i|how do you|what do i do)\b", re.I)
ASK_Q = re.compile(r"(what('?s| is| was) the question|read (it|the question)|say it again|repeat|"
                   r"what are we doing|show me the question|the question again)", re.I)


class ArithTutor:
    """Tutors one Problem. Every number it says is computed in Python; it
    never says the final answer before the kid does; right/wrong is decided
    by exact comparison, never by the language model."""

    def __init__(self, item, problem: Problem, buddy: str = "GEETA"):
        self.item, self.p, self.buddy = item, problem, buddy
        self.steps = plan_steps(problem)
        self.step = 0                 # next step we're waiting on
        self.hint_level = 0
        self.done = False
        self.mistakes: list[dict] = []
        self.wrong_in_a_row = 0
        self.asked_answer = 0

    # -- what the kid sees ------------------------------------------------
    def _ask(self) -> str:
        return self.steps[min(self.step, len(self.steps) - 1)][0]

    def _miss(self, cat, detail=""):
        self.mistakes.append({"cat": cat, "detail": detail})

    def intro(self) -> str:
        opener = pick(["Ooh, a good one!", "Let's do this one together!", "Okay, let's crack it!",
                       "I like this one!"])
        return f"{opener} {self._ask()}"

    def restate(self) -> str:
        return f"Sure! The question is: \"{self.p.text}\" {self._ask()}"

    def nudge(self) -> str:
        return pick(["Take your time! 😊 ", "No rush! ", "I'm right here when you're ready. "]) + self._ask()

    def hint(self) -> str:
        self.hint_level += 1
        p = self.p
        if self.hint_level == 1:
            return f"{pick(STUCK_COMFORT)} {self._ask()}"
        concrete = self._concrete()
        if concrete:
            return concrete
        return f"Let's go super slow. {self._ask()}"

    def _concrete(self) -> Optional[str]:
        """A hands-on way to do the CURRENT step (fingers, drawing, a similar easier one)."""
        p, a, b = self.p, self.p.a, self.p.b
        if p.kind in ("add", "sub"):
            ex = _worked_example(p.extra.get("strategy"), p.answer, sub=p.kind == "sub")
            return f"{ex} Now you try yours! {self._ask()}" if ex else None
        if p.kind == "mul":
            return (f"Try drawing it! Draw {a} circles and put {b} dots in each one. Then count all the dots. "
                    f"How many?") if a * b <= 40 else None
        if p.kind == "div":
            return (f"Try drawing {b} circles, then share out {a} dots one at a time - one in each circle, round and round. "
                    f"How many dots in each circle?") if a <= 40 else None
        return None

    # -- the kid said something -------------------------------------------
    def turn(self, kid_text: str) -> str:
        low = (kid_text or "").lower()
        if ASK_Q.search(low):
            return self.restate()
        nums = _kid_numbers(kid_text)
        if self.p.kind == "oddeven":
            word = "even" if re.search(r"\beven\b", low) else ("odd" if re.search(r"\bodd\b", low) else None)
            if word:
                return self._final(word, word == self.p.answer)
        if self.p.kind in ("letters", "weekday", "limit", "word_pick", "word_open"):
            r = self._word_answer(kid_text, low)
            if r is not None:
                return r
        if self.p.kind == "shape_name":
            named = next((sh for sh in SHAPES if re.search(r"\b" + sh + r"s?\b", low)), None)
            if named:
                if named in self.p.extra["ok"]:
                    return self._final(named, True)
                n_has = SHAPES[named][0]
                self._miss("wrong_answer", f"said {named}")
                return (f"Hmm, a {named} has {n_has} {self.p.extra['what']} - we need {self.p.a}. "
                        f"{'More' if n_has < self.p.a else 'Fewer'} {self.p.extra['what']} than that! Try another shape.")
        if self.p.kind == "two_digits":
            ns = nums
            x, y = self.p.a, self.p.b
            if len(ns) >= 2 and ns[-2] == x and ns[-1] == y:
                return self._final(self.p.answer, True)
            if len(ns) >= 2 and ns[-2] == y and ns[-1] == x and x != y:
                self._miss("digits_swapped")
                return f"You've got both digits - but swapped! Which one is in the {self.p.extra['hi']} place?"
            if len(ns) == 1 and self.step == 0 and ns[0] == x:
                self.step = 1
                return f"{pick(STEP_PRAISE).format(n=x)} {self._ask()}"
            if len(ns) == 1 and self.step == 1 and ns[0] == y:
                return self._final(self.p.answer, True)
            if ns:
                self.wrong_in_a_row += 1
                return f"{pick(TRY_AGAIN)} {self._ask()}"
        if self.p.kind == "fraction":
            m = re.search(r"(\d+)\s*(?:/|out of|over)\s*(\d+)", _words_to_digits(low))
            if m:
                said = f"{int(m.group(1))}/{int(m.group(2))}"
                if said == self.p.answer:
                    return self._final(said, True)
                if int(m.group(1)) == self.p.extra["took"] and self.p.extra["left"]:
                    self._miss("wrong_part", "wrote the eaten part")
                    return "Ooh, that's the part that got EATEN! We want the slices that are LEFT. How many are left?"
                return self._final(said, False)
        if self.p.extra.get("money"):
            said = self._money_said(kid_text)
            if said:
                return self.take(said)
        if nums:
            return self.take(nums)
        if ASK_ANSWER.search(low):
            self.asked_answer += 1
            return f"{pick(NO_SPOILER)} {self._ask()}"
        f = feeling_of(low)
        if f:
            return f"{pick(FEELINGS[f])} {self._ask()}"
        if STUCK.search(low):
            return self.hint()
        return self.nudge()

    def _word_answer(self, kid_text: str, low: str) -> Optional[str]:
        """Answers that are words: a letter, a day, yes/no, or one of the choices."""
        p = self.p
        if p.kind == "letters":
            m = re.findall(r"\b([a-z])\b", low.replace("letter ", ""))
            m = [x for x in m if x not in ("a", "i") or len(low.strip()) <= 2]
            if m:
                said = m[-1].upper()
                if said == p.answer:
                    return self._final(said, True)
                self.wrong_in_a_row += 1
                way = "forwards" if p.b > 0 else "backwards"
                return f"Hmm, not {said}. The letters go {way} - say them out loud from {p.terms[-1]}. Which comes next?"
            return None
        if p.kind == "weekday":
            day = next((d for d in DAYS if d in low or d[:3] == low.strip()[:3] and len(low.strip()) <= 4), None)
            if day:
                return self._final(day.capitalize(), day.capitalize() == p.answer) if day.capitalize() == p.answer else \
                    (self._miss("wrong_answer", f"said {day}") or f"Hmm, not {day.capitalize()}. {self._ask()}")
            return None
        if p.kind == "limit":
            yn = "yes" if re.search(r"\b(yes|yeah|yep|it will|it breaks|break)\b", low) else \
                 ("no" if re.search(r"\b(no|nope|won'?t|will not|it holds|safe)\b", low) else None)
            if yn and self.step >= 1 or (yn and not _kid_numbers(kid_text)):
                if yn == p.answer:
                    self.done = True
                    return (f"{pick(PRAISE)} {yn.capitalize()}! {p.a} is {'more' if p.a > p.b else 'not more'} than {p.b}, "
                            "and YOU worked it out! Want to do another one?")
                self.wrong_in_a_row += 1
                return f"Hmm, let's check: {self._ask()}"
            return None
        if p.kind == "word_pick":
            said = next((o for o in p.terms if re.search(r"\b" + o + r"s?\b", low)), None)
            if said:
                if said.upper() == p.answer:
                    return self._final(said.upper(), True)
                self.wrong_in_a_row += 1
                self._miss("wrong_answer", f"said {said}")
                if p.extra["how"] == "category":
                    return f"Hmm, is a {said} really a {p.extra['cat']}? Have another look at the words!"
                return f"Hmm, {said.upper()} isn't really the other end of {p.extra['target'].upper()}. Try another word!"
            return None
        if p.kind == "word_open":
            said = next((o for o in p.terms if re.search(r"\b" + o + r"s?\b", low)), None)
            if said:
                self.done = True
                self.unchecked = True       # GEETA can't check this one - never pretend to
                return (f"Ooh, {said.upper()} - good thinking! 🤔 Can you say WHY you picked it? "
                        "This is a word puzzle, so I can't check it like maths - ask your grown-up to check it with you. "
                        "Want to do another question?")
            return None
        return None

    def _final(self, said, correct: bool) -> str:
        if correct:
            self.done = True
            shown = fmt_answer(self.p, said) if isinstance(said, int) else said
            return (f"{pick(PRAISE)} {shown} is the answer - and YOU worked it out! "
                    f"Want to do another one?")
        self.wrong_in_a_row += 1
        self._miss("wrong_answer", f"said {said}")
        return f"{pick(TRY_AGAIN)} {self._ask()}"

    def _money_said(self, kid_text: str) -> list[int]:
        """What a kid's money words could mean, in cents: '$2', '2 dollars', '50c', '1.50', or a bare 2 for $2."""
        t = (kid_text or "").lower()
        out = []
        for tok in re.findall(r"\$\s*\d+(?:\.\d{1,2})?|\d+\s*(?:c|cents?)\b", t):
            c = _cents(tok)
            if c is not None:
                out.append(c)
        for d, c in re.findall(r"(\d+) dollars?(?: and (\d+) cents?)?", t):
            out.append(int(d) * 100 + int(c or 0))
        for d, c in re.findall(r"(?<![\d$])(\d+)\.(\d{2})(?!\d)", t):
            out.append(int(d) * 100 + int(c))
        if self.p.extra.get("money") == "dollars":
            out = [c // 100 for c in out if c % 100 == 0]       # "$2" / "2 dollars" -> 2
        if not out:
            out = _kid_numbers(t)                               # a bare "2" or "50"
        return out

    def take(self, nums: list[int]) -> str:
        p = self.p
        if not isinstance(p.answer, int):
            # the answer is a word/fraction - a lone number may be a step (e.g. slices left)
            for i in range(self.step, len(self.steps) - 1):
                if self.steps[i][1] in nums:
                    self.step = i + 1
                    return f"{pick(STEP_PRAISE).format(n=self.steps[i][1])} {self._ask()}"
            self.wrong_in_a_row += 1
            return f"{pick(TRY_AGAIN)} {self._ask()}"
        if p.answer in nums:
            # the answer counts even inside "is it 42?" - but not if it's only part of the question
            return self._final(p.answer, True)
        n = nums[-1]
        # an in-between step they got right
        for i in range(self.step, len(self.steps) - 1):
            if self.steps[i][1] == n:
                self.step = i + 1
                self.wrong_in_a_row = 0
                return f"{pick(STEP_PRAISE).format(n=n)} {self._ask()}"
        self.wrong_in_a_row += 1
        step_expect = self.steps[self.step][1] if self.step < len(self.steps) - 1 else None
        wrong_op = (p.kind == "sub" and n == p.a + p.b) or (p.kind == "add" and n == abs(p.a - p.b)) or \
                   (p.kind == "mul" and n == p.a + p.b)
        if isinstance(step_expect, int) and isinstance(p.answer, int) and not wrong_op and \
                abs(n - step_expect) < abs(n - p.answer):
            # they're answering the little step, not the whole question
            self._miss("step_slip", f"step wanted {step_expect}, said {n}")
            lead = "Ooh, so close - just 1 away! Count again carefully." if abs(n - step_expect) == 1 \
                else pick(TRY_AGAIN)
            if self.wrong_in_a_row >= 2:
                concrete = self._concrete()
                if concrete:
                    return f"{lead} {concrete}"
            return f"{lead} {self._ask()}"
        why = self._diagnose(n)
        if self.wrong_in_a_row >= 2:
            concrete = self._concrete()
            if concrete:
                return f"{why} {concrete}"
        return f"{why} {self._ask()}"

    def _diagnose(self, n: int) -> str:
        """Say what the kid's number tells us - kindly, specifically, never the answer."""
        p, a, b, ans = self.p, self.p.a, self.p.b, self.p.answer
        if not isinstance(ans, int):
            self._miss("wrong_answer", f"said {n}")
            return pick(TRY_AGAIN)
        if p.kind == "add" and n == abs(a - b):
            self._miss("wrong_operation", "took away instead of adding")
            return "Ooh, I think you took away. This one is ADDING - so the answer gets bigger than both numbers."
        if p.kind == "sub" and n == a + b:
            self._miss("wrong_operation", "added instead of taking away")
            return "Ooh, I think you added. This one is TAKING AWAY - so the answer is smaller than " + str(a) + "."
        if p.kind == "mul" and n == a + b:
            self._miss("wrong_operation", "added instead of groups")
            return f"Hmm, that's {a} + {b}. But this is {a} GROUPS of {b} - lots of {b}s, not just one!"
        if p.kind == "mul" and abs(n - ans) == b:
            self._miss("one_group_off")
            return f"So close - I think you counted one group {'too many' if n > ans else 'too few'}. Let's count the groups carefully."
        if abs(n - ans) == 1:
            self._miss("off_by_one")
            return "Ooh, SO close - just 1 away! Count again super carefully."
        if abs(n - ans) == 10:
            self._miss("tens_slip")
            return "Really close! The ones are right - check the tens again."
        if ans >= 10 and str(n) == str(ans)[::-1]:
            self._miss("digits_swapped")
            return "You've got the right digits, but they swapped places! Which one goes in the tens place?"
        if p.kind in ("add", "multi_add") and n < max(a, b, *(p.terms or [0])):
            self._miss("direction")
            return f"Hmm, when we add, the answer gets BIGGER - so it must be more than {max(a, b, *(p.terms or [0]))}."
        if p.kind == "sub" and n > a:
            self._miss("direction")
            return f"Hmm, when we take away, the answer gets SMALLER - so it must be less than {a}."
        self._miss("wrong_answer", f"said {n}")
        return pick(TRY_AGAIN)


# strategy -> several fully worked, DIFFERENT examples (checked by tests: every sum is true).
# We only ever show one containing NO number equal to the kid's answer.
_EXAMPLES = {
    "tens_then_ones": ["Let me show you a different one: 23 + 14. I do 23 + 10 = 33, then 33 + 4 = 37.",
                       "Let me show you a different one: 51 + 26. I do 51 + 20 = 71, then 71 + 6 = 77.",
                       "Let me show you a different one: 12 + 35. I do 35 + 10 = 45, then 45 + 2 = 47."],
    "jump_tens": ["Let me show you a different one: 35 + 20. Jump a ten: 45, jump another: 55.",
                  "Let me show you a different one: 12 + 30. Jump a ten: 22, another: 32, another: 42."],
    "make_ten": ["Let me show you a different one: 8 + 5. 8 needs 2 to make 10, so 5 splits into 2 and 3. 10 + 3 = 13!",
                 "Let me show you a different one: 7 + 6. 7 needs 3 to make 10, so 6 splits into 3 and 3. 10 + 3 = 13!",
                 "Let me show you a different one: 9 + 6. 9 needs 1 to make 10, so 6 splits into 1 and 5. 10 + 5 = 15!"],
    "add_ten_take_one": ["Let me show you a different one: 25 + 9. 25 + 10 = 35, take 1 away: 34.",
                         "Let me show you a different one: 41 + 9. 41 + 10 = 51, take 1 away: 50."],
    "count_on": ["Here's how I count on: 6 + 3. I start at 6 and say 7, 8, 9 - so it's 9.",
                 "Here's how I count on: 12 + 2. I start at 12 and say 13, 14 - so it's 14."],
    "count_up": ["Let me show you a different one: from 5 to 9 I count 6, 7, 8, 9 - that's 4 jumps.",
                 "Let me show you a different one: from 12 to 15 I count 13, 14, 15 - that's 3 jumps."],
    "back_to_ten": ["Let me show you a different one: 12 - 5. Take 2 to get to 10, then 3 more: 10 - 3 = 7.",
                    "Let me show you a different one: 14 - 6. Take 4 to get to 10, then 2 more: 10 - 2 = 8."],
    "take_ten_add_one": ["Let me show you a different one: 34 - 9. 34 - 10 = 24, give 1 back: 25.",
                         "Let me show you a different one: 52 - 9. 52 - 10 = 42, give 1 back: 43."],
    "count_back": ["Here's how I count back: 9 - 3. I start at 9 and say 8, 7, 6 - so it's 6.",
                   "Here's how I count back: 15 - 2. I start at 15 and say 14, 13 - so it's 13."],
    "place_parts": ["Let me show you a different one: 214 + 123. 214 + 100 = 314, then 314 + 20 = 334, then 334 + 3 = 337.",
                    "Let me show you a different one: 531 + 246. 531 + 200 = 731, then 731 + 40 = 771, then 771 + 6 = 777."],
}
_EX_SUB = {
    "tens_then_ones": ["Let me show you a different one: 47 - 13. I do 47 - 10 = 37, then 37 - 3 = 34.",
                       "Let me show you a different one: 86 - 24. I do 86 - 20 = 66, then 66 - 4 = 62.",
                       "Let me show you a different one: 58 - 31. I do 58 - 30 = 28, then 28 - 1 = 27."],
    "jump_tens": ["Let me show you a different one: 75 - 20. Jump back a ten: 65, and another: 55.",
                  "Let me show you a different one: 93 - 30. Jump back a ten: 83, another: 73, another: 63."],
    "place_parts": ["Let me show you a different one: 365 - 142. 365 - 100 = 265, then 265 - 40 = 225, then 225 - 2 = 223.",
                    "Let me show you a different one: 879 - 456. 879 - 400 = 479, then 479 - 50 = 429, then 429 - 6 = 423."],
}


def _worked_example(strategy, answer, sub=False):
    """A worked example for this strategy that contains NO number equal to the
    kid's answer (so it can't give the answer away by accident)."""
    for ex in (_EX_SUB if sub else {}).get(strategy, []) + ([] if sub and strategy in _EX_SUB else _EXAMPLES.get(strategy, [])):
        if str(answer) not in re.findall(r"\d+", ex):
            return ex
    return None


def _kid_numbers(t: str) -> list[int]:
    t = re.sub(r"(?<!\d)(\d{1,2}):(\d{2})(?!\d)", r"\1\2", t or "")       # voice "4:50" -> 450
    t = _words_to_digits(t.lower())
    t = re.sub(r"(\d+) hundred (?:and )?(\d+)", lambda m: str(int(m.group(1)) * 100 + int(m.group(2))), t)
    t = re.sub(r"(\d+) hundred", lambda m: str(int(m.group(1)) * 100), t)
    t = re.sub(r"(\d+)\s*dollars?(?: and (\d+) cents?)?", lambda m: str(int(m.group(1)) * 100 + int(m.group(2) or 0)), t)
    t = re.sub(r"\$\s*(\d+)\.(\d{2})", lambda m: str(int(m.group(1)) * 100 + int(m.group(2))), t)
    return [int(x) for x in re.findall(r"(?<![\d.])\d+(?!\d|\.\d)", t)]


# ---------------------------------------------------------------------------
# Small talk: friendly, honest, short - then back to maths. No numbers made up.
# ---------------------------------------------------------------------------

JOKES = ["Why was 6 scared of 7? Because 7 8 9! 😂",
         "Why did the maths book look sad? It had too many problems! 📚",
         "What do you call a number that can't sit still? A roamin' numeral! 😄",
         "What did the zero say to the eight? Nice belt! 😎",
         "Why did the student wear glasses in maths class? To improve di-vision! 👓",
         "What's a maths teacher's favourite place? Times Square! 🗽"]
BUDDY_KIND = {"Bobo": "teddy bear", "Zoom": "race car", "Cosmo": "rocket", "Rexy": "dinosaur",
              "Luna": "unicorn", "Ollie": "octopus", "GEETA": "maths buddy"}
BUDDY_LIKES = {"Bobo": "honey sandwiches and puzzles", "Zoom": "going super fast and counting laps",
               "Cosmo": "stars - there are SO many to count", "Rexy": "stomping and big numbers",
               "Luna": "rainbows and sparkly patterns", "Ollie": "swimming and counting with all eight arms",
               "GEETA": "puzzles and helping kids"}


PERSONAL = re.compile(r"(\b\d{8,}\b|\b04\d{2}\s?\d{3}\s?\d{3}\b|\+61|@[a-z0-9-]+\.[a-z]|\b\d+ [a-z]+ (street|st|road|rd|avenue|ave|court|ct|drive|dr|lane|ln|place|pl)\b|"
                      r"\bmy (address|phone|number is|password|email|last name|surname|school is)\b|\bi live (at|on|in)\b)", re.I)
RUDE = re.compile(r"\b(stupid|dumb|idiot|shut up|hate you|you suck|loser|ugly)\b", re.I)
MATHY = re.compile(r"\d|plus|minus|times|divide|share|add|take away|how many|how much|total|altogether|left|"
                   r"more|less|fewer|double|half|quarter|fraction|shape|sides|corners|clock|o'clock|time|minutes|"
                   r"hours|money|cents|dollars|count|number|pattern|next|odd|even|round|tens|ones|hundreds|sum|"
                   r"difference|equals|answer|question|maths|math|homework", re.I)


def is_personal_info(text: str) -> bool:
    return bool(PERSONAL.search(text or ""))


def small_talk(text: str, buddy: str = "GEETA", kid: str = "") -> Optional[str]:
    """A friendly reply to chit-chat, or None if it isn't chit-chat."""
    t = (text or "").lower().strip()
    if not t:
        return None
    if is_personal_info(t):
        return ("Thanks for trusting me! 💛 But things like addresses, phone numbers and passwords are private - "
                "only share them with grown-ups you know. Let's keep going with maths!")
    if re.search(r"\d", t):
        return None
    if RUDE.search(t):
        return pick(["Ouch! 😅 I'm still learning too. Let's be kind to each other - ready to try some maths?",
                     "Hmm, that's not very kind. 🙂 Maybe you're feeling frustrated? That's okay - let's take a breath and try together."])
    if re.search(r"\b(i hate maths|i hate math|maths is boring|math is boring|i don'?t like maths)\b", t):
        return pick(["Aw, maths can feel tricky sometimes. 💛 But I bet we can make this one fun - let's do it together!",
                     "I get it - some days maths feels like a mountain. ⛰️ Let's climb just one little step."])
    if re.fullmatch(r"(lol|haha+|hehe+|lmao|😂+|🤣+)[!.\s]*", t):
        return pick(["😄 Hehe! Okay, let's get back to it!", "Haha! 😆 Right, maths time!"])
    who = f"{buddy} the {BUDDY_KIND.get(buddy, 'maths buddy')}"
    you = f" {kid}" if kid and kid.lower() != "kid" else ""
    if re.fullmatch(r"(hi+|hello+|hey+|hiya|g'?day|good (morning|afternoon|evening))[\s!.]*(there|" + re.escape(buddy.lower()) + r")?[\s!.]*", t):
        return pick([f"Hi{you}! 👋 So happy you're here. What maths are we doing today?",
                     f"Hey{you}! 😊 Ready for some maths fun? Show me your homework!"])
    if re.search(r"how are you|how r u|how'?s it going|how do you feel", t):
        return pick(["I'm great, thanks for asking! 😊 Even better now you're here. How are YOU?",
                     "I'm super, thank you! 🌟 Shall we do some maths together?"])
    if re.search(r"what'?s your name|who are you|what are you called|your name", t):
        return f"I'm {who}! 😊 I'm here to help you work out maths yourself - I give clues, not answers."
    if re.search(r"are you (a )?(robot|real|human|computer|ai|person|alive)|you'?re a robot", t):
        return (f"I'm a computer helper called {buddy} - not a real {BUDDY_KIND.get(buddy, 'person')}, "
                "but I really do love maths! 😄 A grown-up made me to help kids learn.")
    if re.search(r"joke|make me laugh|something funny", t):
        return pick(JOKES) + " Okay, back to maths!"
    if re.search(r"what do you like|favou?rite|do you like", t):
        return f"I love {BUDDY_LIKES.get(buddy, 'puzzles')}! 😄 And maths with you, of course."
    if re.search(r"\b(thank(s| you)|ta|cheers)\b", t):
        return pick(["You're welcome! 😊", "Any time! You're doing great.", "My pleasure! 🌟"])
    if re.search(r"\b(bye|goodbye|see you|good ?night|i'?m done|gotta go)\b", t):
        return pick([f"Bye{you}! 👋 You worked really hard today - be proud!",
                     f"See you next time{you}! 🌟 Great thinking today."])
    if re.search(r"\b(i love you|you'?re (nice|cool|the best|funny))\b", t):
        return pick(["Aww, thank you! 💛 You're pretty awesome too.", "That made my day! 😊"])
    if re.search(r"\b(yes|yeah|yep|ok|okay|sure|ready)\b", t) and len(t) < 15:
        return None   # let the tutor carry on
    # a question about something that isn't maths ("capital of France?", "help with spelling")
    if (t.endswith("?") or re.match(r"(what|who|where|why|when|can you|could you|will you|do you know)\b", t)) \
            and not MATHY.search(t):
        return pick([f"Ooh, good question! But I'm {buddy}, a MATHS buddy - I only know about numbers. 😄 "
                     "A grown-up or your teacher can help with that one. Got a maths question for me?",
                     "That's not a maths question, so I'm not the best helper for it! 😊 "
                     "Try asking a grown-up. But if you've got some maths, I'm ready!"])
    return None
